# 04 — Localizar grandes usuários (CEPs com sufixo 900 ou mais)

**Problema:** CEPs de grande usuário (sufixo 900 a 999: empresas, órgãos, condomínios) não têm rua nos Correios nem no Geosanja e ficariam sem ponto (classe E). Eles concentram muitos vínculos (ex.: Embraer, 12227-901).

**Solução:** para cada CEP de grande usuário **do município** (prefixo 12) com vínculos em **algum ano** da RAIS:
1. **ViaCEP** → nome do estabelecimento (`unidade`), rua (`logradouro`) e número (`complemento`);
2. **CNEFE** → coordenada, nesta ordem de confiança: `cnefe_cep_numero` (mesmo CEP e número) · `cnefe_endereco` (mesma rua e número) · `cnefe_cep` (mediana dos endereços com o CEP) · `cnefe_rua_numero_proximo` (mesma rua, número mais próximo);
3. **BrasilAPI** só se o CNEFE falhar → `api_aproximado` (confira);
4. o resto fica sem coordenada para você preencher (`fonte_coord = manual`).

**CEP substituído:** se o Busca CEP mostrar que o CEP mudou, informe o novo na coluna `cep_novo` do CSV (ou em `grandes_usuarios.ceps_substituidos` no `config.yaml`). As consultas usam o CEP novo; o ponto fica no CEP antigo (o da RAIS).

**Saída (editável):** `02_dados_editados\grande_usuario_ceps.csv` (separador `;`). Funciona como **cache**: linhas já localizadas ou `manual` não são refeitas. Antes de regravar, cópia em `02_dados_editados\_backup\`. Edite em editor de texto ou QGIS (no Excel em português, `-23.17944` pode virar data).

**Entradas:** RAIS consolidada (etapa 01), CNEFE (`01_dados_brutos\CNEFE`), internet (ViaCEP e BrasilAPI).
**Usado por:** etapa 05 (cria a classe G) e etapa 06 (pontos na camada da RAIS).

**Estrutura:** 1 Preparação · 2 Funções · 3 CEPs alvo e CSV existente · 4 ViaCEP · 5 CNEFE · 6 Localizar · 7 Exportação · 8 Resumo

## 1. Preparação e configuração

In [ ]:
# --- Preparação padrão: acha a raiz do projeto, lê o config.yaml e liga o log (tela + arquivo) -------------
import sys                                            # para o Python achar as funções comuns (scripts/src)
from pathlib import Path                              # caminhos de arquivos
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())   # sobe pastas até achar o config.yaml
sys.path.insert(0, str(RAIZ / "scripts"))             # libera o "from src import ..."
from src import config, registro                      # funções comuns do padrão de projetos
cfg = config.carregar(RAIZ)                           # lê o config.yaml (caminhos e parâmetros)
ETAPA = "04_grandes_usuarios"               # nome desta etapa (= nome do notebook)
log = registro.iniciar_log(ETAPA, cfg)                # log na tela e em 03_processamento/logs/
PASTA_SAIDA = cfg.pasta_etapa(ETAPA)                  # 03_processamento/<ETAPA> (criada se não existir)
import re                             # busca de padrões em texto
import time                           # pausas entre consultas
import numpy as np                    # contas com colunas
import pandas as pd                   # tabelas
import requests                       # consultas às APIs
from src import fontes                # arquivos editados (prioridade e backup)

# --- Entradas e saída (config.yaml) ----------------------------------------------------------------
ARQ_RAIS = cfg.caminho("processamento.rais_consolidada")     # rais consolidada (etapa 01)
ARQ_CNEFE = cfg.caminho("brutos.cnefe")                      # cnefe do município
ARQ_GRANDES = cfg.caminho("editados.grandes_usuarios")       # CSV editável (02_dados_editados)

# --- Parâmetros ----------------------------------------------------------------------------------------
PREFIXO_MUNICIPIO = cfg.projeto("prefixo_cep")                             # só CEPs do município
SUFIXO_ESPECIAL_MIN = cfg.parametro("grandes_usuarios.sufixo_especial_min")   # sufixo >= isso = grande usuário
URL_VIACEP = "https://viacep.com.br/ws/{cep}/json/"                        # nome, rua e número do estabelecimento
URL_BRASILAPI = "https://brasilapi.com.br/api/cep/v2/{cep}"                # coordenada aproximada (só se o cnefe falhar)
PAUSA_S = cfg.parametro("grandes_usuarios.pausa_s")                        # pausa entre consultas
TIMEOUT_S = cfg.parametro("grandes_usuarios.timeout_s")                    # tempo máximo de espera por resposta
TENTATIVAS = cfg.parametro("grandes_usuarios.tentativas")                  # tentativas em caso de erro de rede
MARGEM_GRAUS = cfg.parametro("grandes_usuarios.margem_graus")              # folga ao redor da área do cnefe para aceitar a API
CEPS_SUBSTITUIDOS = {str(k): str(v) for k, v in (cfg.parametro("grandes_usuarios.ceps_substituidos") or {}).items()}   # antigo -> novo
COLUNAS_CNEFE = ["CEP", "NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR", "NUM_ENDERECO",
                 "LATITUDE", "LONGITUDE", "DSC_ESTABELECIMENTO"]
COLUNAS_CSV = ["cep", "cep_novo", "unidade", "logradouro", "numero", "bairro", "status_consulta", "lat", "lon", "fonte_coord",
               "n_pontos_cnefe", "obs", "vinc_ativos_soma_anos"]

log.info("Configuração carregada!")
for _nome, _arq in [("rais consolidada", ARQ_RAIS), ("cnefe", ARQ_CNEFE), ("CSV de grandes usuários", ARQ_GRANDES)]:
    log.info("%s: %s | existe: %s", _nome, _arq.name, _arq.exists())

## 2. Funções auxiliares

In [ ]:
from src.validacao import validar_tabela                         # bloco padrão de validação
from src.texto import normalizar, normalizar_nome                # normalização de nomes de logradouro


def consultar_json(url):
    """Faz GET em uma API com tentativas. Devolve (dados, status): status = ok | nao_encontrado | erro_rede."""
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = requests.get(url, timeout=TIMEOUT_S, headers={"User-Agent": "projeto-rais"})
            if resposta.status_code == 404:
                return None, "nao_encontrado"
            resposta.raise_for_status()                 # erro HTTP vira exceção
            return resposta.json(), "ok"
        except Exception as erro:                        # rede caiu, tempo esgotado etc.
            log.warning("  falha (tentativa %d/%d) em %s: %s", tentativa, TENTATIVAS, url, erro)
            time.sleep(PAUSA_S * tentativa)
    return None, "erro_rede"


def numero_do_texto(texto):
    """Primeiro número inteiro de um texto (ex.: '2170' -> 2170). Sem número devolve None."""
    achado = re.search(r"\d+", str(texto or ""))
    return int(achado.group()) if achado else None


log.info("Funções auxiliares definidas!")

## 3. CEPs alvo e CSV existente
- **Alvo:** CEPs de grande usuário (sufixo ≥ 900) com vínculos em **algum ano** (soma dos 9 anos > 0), que estão nas classes **E** ou **G** da Etapa 05 (G = já localizados antes);
- **CSV existente:** linhas com coordenada (ou marcadas `manual`) são mantidas; as demais são refeitas.

In [ ]:
log.info("Selecionando os CEPs de grande usuário na rais...")
rais = pd.read_parquet(ARQ_RAIS, columns=["cep", "qtd_vinc_ativos"])
rais["cep"] = rais["cep"].astype(object).where(rais["cep"].notna(), "").astype(str)
por_cep = (rais.groupby("cep", as_index=False)
           .agg(vinc_ativos_soma_anos=("qtd_vinc_ativos", lambda s: int(s.fillna(0).sum()))))   # vínculos somando os anos
sufixo = pd.to_numeric(por_cep["cep"].str[-3:], errors="coerce")
alvos = por_cep[por_cep["cep"].str.fullmatch(r"\d{8}") & por_cep["cep"].str.startswith(PREFIXO_MUNICIPIO)
                & (sufixo >= SUFIXO_ESPECIAL_MIN) & (por_cep["vinc_ativos_soma_anos"] > 0)].copy()
alvos = alvos.sort_values("vinc_ativos_soma_anos", ascending=False).reset_index(drop=True)
log.info("CEPs de grande usuário do município com vínculos: %d | vínculos (soma dos anos): %d",
         len(alvos), int(alvos["vinc_ativos_soma_anos"].sum()))

if ARQ_GRANDES.exists():
    existente = pd.read_csv(ARQ_GRANDES, sep=";", dtype=str, encoding="utf-8-sig", keep_default_na=False)
    for col in COLUNAS_CSV:
        if col not in existente.columns:
            existente[col] = ""
    mantidas = existente[(existente["fonte_coord"] == "manual") | (existente["lat"].str.strip() != "")].copy()
    log.info("CSV existente: %d linhas | mantidas (com coordenada ou manual): %d", len(existente), len(mantidas))
else:
    existente = pd.DataFrame(columns=COLUNAS_CSV)
    mantidas = existente.copy()
    log.info("CSV ainda não existe: será criado.")
# --- CEPs substituídos: coluna cep_novo do CSV (prevalece) + dicionário da configuração ----------------------------------------
novo_csv = existente.set_index("cep")["cep_novo"].str.replace(r"\D", "", regex=True) if len(existente) else pd.Series(dtype=str)
mapa_novo = {**CEPS_SUBSTITUIDOS, **{k: v for k, v in novo_csv.items() if len(v) == 8}}      # só valores com 8 dígitos
if len(existente):
    marcador = existente["cep"].map(lambda c: f"CEP substituído por {mapa_novo[c]}" if c in mapa_novo else None)
    ja_refeito = pd.Series([isinstance(m, str) and o.startswith(m) for m, o in zip(marcador, existente["obs"])], index=existente.index)
    reprocessar = existente["cep"].isin(list(mapa_novo)) & (existente["fonte_coord"] != "manual") & ~ja_refeito
    mantidas = mantidas[~mantidas["cep"].isin(existente.loc[reprocessar, "cep"])]            # refaz quem ganhou CEP novo
log.info("CEPs com CEP novo informado: %d", len(mapa_novo))
pendentes = alvos[~alvos["cep"].isin(mantidas["cep"])].copy()
pendentes["cep_busca"] = pendentes["cep"].map(lambda c: mapa_novo.get(c, c))                  # CEP usado nas consultas
log.info("CEPs a consultar agora: %d", len(pendentes))
validar_tabela("CEPs alvo", alvos, chave=["cep"], obrigatorias=["cep"])
log.info("CEPs alvo prontos!")

## 4. Consulta ao ViaCEP
Busca nome, rua, número e bairro de cada CEP pendente (uma consulta por CEP, com pausa). Se a rede falhar, o CEP fica com `status_consulta = erro_rede` e é refeito na próxima execução.

In [ ]:
log.info("Consultando o ViaCEP (%d CEPs)...", len(pendentes))
registros = []
for i, (cep, busca) in enumerate(zip(pendentes["cep"], pendentes["cep_busca"]), start=1):   # laço curto, limitado pela rede
    dados, status = consultar_json(URL_VIACEP.format(cep=busca))
    if status == "ok" and (not isinstance(dados, dict) or dados.get("erro")):
        status = "nao_encontrado"                              # o ViaCEP responde {"erro": true} quando não acha
    d = dados if status == "ok" else {}
    registros.append({"cep": cep, "cep_busca": busca, "unidade": d.get("unidade", ""), "logradouro": d.get("logradouro", ""),
                      "numero": d.get("complemento", ""), "bairro": d.get("bairro", ""), "status_consulta": status})
    if i % 10 == 0 or i == len(pendentes):
        log.info("  %d/%d consultados", i, len(pendentes))
    time.sleep(PAUSA_S)
consulta = pd.DataFrame(registros, columns=["cep", "cep_busca", "unidade", "logradouro", "numero", "bairro", "status_consulta"])
log.info("Resultado da consulta:\n%s", consulta["status_consulta"].value_counts().to_string() if len(consulta) else "(nada a consultar)")
log.info("Consulta ao ViaCEP concluída!")

## 5. Carregar o CNEFE
Lê só as colunas necessárias, converte as coordenadas e guarda, para cada endereço, o CEP, o nome normalizado da rua e o número. Endereços sem número (`0`, `SN`) ficam sem número.

In [ ]:
log.info("Carregando o cnefe...")
def ler_cnefe(codificacao):
    """Lê o cnefe (separador ';') só com as colunas necessárias."""
    return pd.read_csv(ARQ_CNEFE, sep=";", dtype=str, usecols=COLUNAS_CNEFE, encoding=codificacao, keep_default_na=False)

try:
    cnefe = ler_cnefe("utf-8")
except UnicodeDecodeError:                                       # arquivo em Latin-1
    log.warning("Arquivo não está em UTF-8; relendo como Latin-1.")
    cnefe = ler_cnefe("latin-1")
n_cnefe = len(cnefe)
cnefe["lat"] = pd.to_numeric(cnefe["LATITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe["lon"] = pd.to_numeric(cnefe["LONGITUDE"].str.replace(",", ".", regex=False), errors="coerce")
cnefe = cnefe[cnefe["lat"].notna() & cnefe["lon"].notna()].copy()
cnefe["cep"] = cnefe["CEP"].str.replace(r"\D", "", regex=True)
cnefe["num"] = pd.to_numeric(cnefe["NUM_ENDERECO"], errors="coerce").where(lambda s: s > 0)     # 0 = sem número

# --- Nome da rua normalizado (só nas combinações distintas: rápido) -----------------------------------------------------------
ruas = cnefe[["NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"]].drop_duplicates().copy()
ruas["rua_key"] = normalizar(ruas["NOM_TIPO_SEGLOGR"] + " " + ruas["NOM_TITULO_SEGLOGR"] + " " + ruas["NOM_SEGLOGR"])
cnefe = cnefe.merge(ruas, on=["NOM_TIPO_SEGLOGR", "NOM_TITULO_SEGLOGR", "NOM_SEGLOGR"], how="left")
cnefe = cnefe[["cep", "rua_key", "num", "lat", "lon", "DSC_ESTABELECIMENTO"]]
BBOX = (cnefe["lon"].min() - MARGEM_GRAUS, cnefe["lat"].min() - MARGEM_GRAUS, cnefe["lon"].max() + MARGEM_GRAUS, cnefe["lat"].max() + MARGEM_GRAUS)
log.info("cnefe: %d endereços lidos | %d com coordenada | com número: %d | área (lon/lat): %s",
         n_cnefe, len(cnefe), int(cnefe["num"].notna().sum()), tuple(round(float(v), 3) for v in BBOX))
validar_tabela("cnefe", cnefe, obrigatorias=["cep", "rua_key", "lat", "lon"], linhas_entrada=n_cnefe)
log.info("cnefe carregado!")

## 6. Localizar
Para cada CEP consultado, tenta as fontes na ordem de confiança (ver topo). A BrasilAPI só é chamada se o CNEFE não resolver, e sua coordenada só vale se estiver dentro da área do CNEFE (descarta pontos no meio do mapa ou fora do município).

In [ ]:
log.info("Localizando as coordenadas...")
ceps_alvo_cnefe = set(pendentes["cep_busca"])
cnefe_cep = {k: v for k, v in cnefe[cnefe["cep"].isin(ceps_alvo_cnefe)].groupby("cep")}     # endereços por CEP
rua_keys = {normalizar_nome(r) for r in consulta["logradouro"] if r}
cnefe_rua = {k: v for k, v in cnefe[cnefe["rua_key"].isin(rua_keys)].groupby("rua_key")}    # endereços por rua


def localizar(cep, logradouro, numero_txt):
    """Devolve (lat, lon, fonte, n_pontos, obs) para um CEP, tentando as fontes em ordem de confiança."""
    num = numero_do_texto(numero_txt)
    por_cep = cnefe_cep.get(cep)
    por_rua = cnefe_rua.get(normalizar_nome(logradouro)) if logradouro else None
    # 1) mesmo CEP e mesmo número
    if por_cep is not None and num is not None:
        m = por_cep[por_cep["num"] == num]
        if len(m):
            return m["lat"].mean(), m["lon"].mean(), "cnefe_cep_numero", len(m), ""
    # 2) mesma rua e mesmo número
    if por_rua is not None and num is not None:
        m = por_rua[por_rua["num"] == num]
        if len(m):
            return m["lat"].mean(), m["lon"].mean(), "cnefe_endereco", len(m), ""
    # 3) endereços do cnefe com esse CEP (mediana dos pontos)
    if por_cep is not None:
        return por_cep["lat"].median(), por_cep["lon"].median(), "cnefe_cep", len(por_cep), \
            "vários endereços com o CEP: confira" if len(por_cep) > 1 else ""
    # 4) mesma rua, número mais próximo
    if por_rua is not None and num is not None and por_rua["num"].notna().any():
        dif = (por_rua["num"] - num).abs()
        melhor = por_rua[dif == dif.min()]
        return melhor["lat"].mean(), melhor["lon"].mean(), "cnefe_rua_numero_proximo", len(melhor), \
            f"número {num} não existe no cnefe; usado o {int(melhor['num'].iloc[0])}"
    # 5) BrasilAPI (aproximada)
    dados, status = consultar_json(URL_BRASILAPI.format(cep=cep))
    time.sleep(PAUSA_S)
    try:
        coord = dados["location"]["coordinates"]
        lat, lon = float(coord["latitude"]), float(coord["longitude"])
        if BBOX[0] <= lon <= BBOX[2] and BBOX[1] <= lat <= BBOX[3]:
            return lat, lon, "api_aproximado", 0, "coordenada da API (aproximada): confira"
        return np.nan, np.nan, "sem_coordenada", 0, "coordenada da API fora da área do município (descartada)"
    except Exception:
        return np.nan, np.nan, "sem_coordenada", 0, "sem coordenada: preencher lat/lon e fonte_coord = manual"


resultados = []
for i, linha in enumerate(consulta.itertuples(index=False), start=1):            # laço curto (só os CEPs pendentes)
    if linha.status_consulta == "ok":
        lat, lon, fonte, n_pts, obs = localizar(linha.cep_busca, linha.logradouro, linha.numero)
    else:
        lat, lon, fonte, n_pts, obs = np.nan, np.nan, "sem_coordenada", 0, f"consulta ao ViaCEP: {linha.status_consulta}"
    if linha.cep_busca != linha.cep:                                  # avisa que a busca usou o CEP novo
        obs = f"CEP substituído por {linha.cep_busca}" + (f"; {obs}" if obs else "")
    resultados.append({"cep": linha.cep, "lat": lat, "lon": lon, "fonte_coord": fonte, "n_pontos_cnefe": n_pts, "obs": obs})
    if i % 10 == 0 or i == len(consulta):
        log.info("  %d/%d localizados", i, len(consulta))
novos = consulta.merge(pd.DataFrame(resultados, columns=["cep", "lat", "lon", "fonte_coord", "n_pontos_cnefe", "obs"]), on="cep", how="left")
anotacao_antiga = existente.drop_duplicates("cep").set_index("cep")["cep_novo"] if len(existente) else pd.Series(dtype=str)
novos["cep_novo"] = novos["cep"].map(mapa_novo).fillna(novos["cep"].map(anotacao_antiga)).fillna("")      # mantém anotações como "nao_encontrado"
log.info("Fontes das coordenadas (novos):\n%s", novos["fonte_coord"].value_counts().to_string() if len(novos) else "(nada novo)")
log.info("Localização concluída!")

## 7. Exportação
Junta as linhas mantidas com as novas, atualiza os vínculos e grava o CSV (ordenado por vínculos). Se o CSV já existia, guarda antes uma **cópia de segurança** com data e hora, para não perder edições suas.

In [ ]:
log.info("Gravando o CSV de grandes usuários...")
for col in ["lat", "lon"]:
    novos[col] = novos[col].map(lambda v: "" if pd.isna(v) else f"{v:.6f}")          # texto com ponto decimal
novos["n_pontos_cnefe"] = novos["n_pontos_cnefe"].astype(str)
final = pd.concat([mantidas[COLUNAS_CSV].astype(str), novos.reindex(columns=COLUNAS_CSV).fillna("").astype(str)], ignore_index=True)
final = final.drop(columns=["vinc_ativos_soma_anos"]).merge(alvos[["cep", "vinc_ativos_soma_anos"]], on="cep", how="left")
final["vinc_ativos_soma_anos"] = final["vinc_ativos_soma_anos"].fillna(0).astype(int)
final = final.sort_values("vinc_ativos_soma_anos", ascending=False)[COLUNAS_CSV].reset_index(drop=True)

if ARQ_GRANDES.exists() and len(novos) == 0:
    log.info("Nada novo: o CSV existente foi mantido sem alterações.")
else:
    if ARQ_GRANDES.exists():
        fontes.backup(cfg, ARQ_GRANDES)                        # cópia datada em 02_dados_editados/_backup (pode ter edições manuais)
    ARQ_GRANDES.parent.mkdir(parents=True, exist_ok=True)
    final.to_csv(ARQ_GRANDES, sep=";", index=False, encoding="utf-8-sig")
    log.info("CSV salvo: %s (%d linhas)", ARQ_GRANDES.name, len(final))
validar_tabela("CSV de grandes usuários", final, chave=["cep"], obrigatorias=["cep"])
log.info("Exportação concluída!")

## 8. Resumo

In [ ]:
log.info("Resumo final...")
com_coord = final["lat"].str.strip() != ""
vinc_total = int(final["vinc_ativos_soma_anos"].sum())
vinc_loc = int(final.loc[com_coord, "vinc_ativos_soma_anos"].sum())
log.info("CEPs de grande usuário: %d | com coordenada: %d | sem coordenada: %d", len(final), int(com_coord.sum()), int((~com_coord).sum()))
log.info("Vínculos (soma dos anos) com coordenada: %d de %d (%.1f%%)", vinc_loc, vinc_total, vinc_loc / vinc_total * 100 if vinc_total else 0.0)
log.info("Por fonte:\n%s", final["fonte_coord"].value_counts().to_string())
conferir = final[final["fonte_coord"].isin(["api_aproximado", "cnefe_rua_numero_proximo"]) | final["obs"].str.contains("confira")]
log.info("CEPs para você conferir (aproximados): %d", len(conferir))
sem = final[~com_coord].head(10)
if len(sem):
    log.info("Maiores sem coordenada (preencher lat/lon e fonte_coord = manual):\n%s", sem[["cep", "unidade", "logradouro", "numero", "vinc_ativos_soma_anos"]].to_string(index=False))
log.info("Próximo: etapas 05 (cria a classe G) e 06 (pontos na camada da rais).")
log.info("Etapa 04 concluída!")